In [ ]:
def order_jobs_in_descending_order_of_total_completion_time(processing_times):
    total_completion_time = processing_times.sum(axis=1)
    return np.argsort(total_completion_time, axis=0).tolist()

In [ ]:
def insertion(sequence, position, value):
    new_seq = sequence[:]
    new_seq.insert(position, value)
    return new_seq

In [ ]:
def evaluate_sequence(sequence, processing_times):
    _, num_machines = processing_times.shape
    num_jobs = len(sequence)

    # Check if the sequence is empty
    if num_jobs == 0:
        # Return a default value (you may choose 0 or another suitable value)
        return 0

    completion_times = np.zeros((num_jobs, num_machines))

    # Calculate the completion times for the first machine
    completion_times[0][0] = processing_times[sequence[0]][0]
    for i in range(1, num_jobs):
        completion_times[i][0] = completion_times[i-1][0] + processing_times[sequence[i]][0]

    # Calculate the completion times for the remaining machines
    for j in range(1, num_machines):
        completion_times[0][j] = completion_times[0][j-1] + processing_times[sequence[0]][j]
        for i in range(1, num_jobs):
            completion_times[i][j] = max(completion_times[i-1][j], completion_times[i][j-1]) + processing_times[sequence[i]][j]

    # Return the total completion time, which is the completion time of the last job in the last machine
    return completion_times[num_jobs-1][num_machines-1]


In [ ]:
def deviation(Cmax, UP):
    return ((Cmax - UP)/UP)*100

# Heuristics

In [ ]:
def neh_algorithm(processing_times):
    ordered_sequence = order_jobs_in_descending_order_of_total_completion_time(processing_times)
    # Define the initial order
    J1, J2 = ordered_sequence[:2]
    sequence = [J1, J2] if evaluate_sequence([J1, J2], processing_times) < evaluate_sequence([J2, J1], processing_times) else [J2, J1]
    del ordered_sequence[:2]
    # Add remaining jobs
    for job in ordered_sequence:
        Cmax = float('inf')
        best_sequence = []
        for i in range(len(sequence)+1):
            new_sequence = insertion(sequence, i, job)
            Cmax_eval = evaluate_sequence(new_sequence, processing_times)
            if Cmax_eval < Cmax:
                Cmax = Cmax_eval
                best_sequence = new_sequence
        sequence = best_sequence
    return sequence, Cmax

In [ ]:
def ham_heuristic(processing_time):
    jobs, machines = processing_time.shape
    sequence = list(range(jobs))
    # Calculating the first summation
    P1 = processing_time[:,:machines//2].sum(axis=1)
    # Calculating the second summation
    P2 = processing_time[:,machines//2:].sum(axis=1)
    # Calculating the first solution, ordered by P2 - P1
    P2_P1 = P2 - P1
    solution_1 = [job for _ , job in sorted(zip(P2_P1, sequence), reverse=True)]
    # Calculating the second solution
    positives = np.argwhere(P2_P1 >= 0).flatten()
    negatives = np.argwhere(P2_P1 < 0).flatten()
    positive_indices = [job for _ , job in sorted(zip(P1[positives], positives))]
    negative_indices = [job for _ , job in sorted(zip(P2[negatives], negatives), reverse=True)]
    positive_indices.extend(negative_indices)
    # Calculating Cmax for both solutions
    Cmax1 = evaluate_sequence(solution_1, processing_time)
    Cmax2 = evaluate_sequence(positive_indices, processing_time)
    # Returning the best solution among them
    if Cmax1 < Cmax2:
        return solution_1, Cmax1
    else:
        return positive_indices, Cmax2

In [ ]:
def palmer_heuristic(processing_times):
    jobs, machines = processing_times.shape
    slope_indices = []
    for i in range(jobs):
        processing_time_sum = np.sum(processing_times[i])
        fi = 0
        for j in range(machines):
            fi += (machines - 2*j + 1) * processing_times[i][j] / processing_time_sum
        slope_indices.append(fi)
    order = sorted(range(jobs), key=lambda k: slope_indices[k])
    return order

In [ ]:
def johnson_method(processing_times):
    jobs, machines = processing_times.shape
    copy_processing_times = processing_times.copy()
    maximum = processing_times.max() + 1
    m1 = []
    m2 = []

    if machines != 2:
        raise Exception("Johson method only works with two machines")

    for i in range(jobs):
        minimum = copy_processing_times.min()
        position = np.where(copy_processing_times == minimum)

        if position[1][0] == 0:
            m1.append(position[0][0])
        else:
            m2.insert(0, position[0][0])

        copy_processing_times[position[0][0]] = maximum

    return m1+m2

In [ ]:
def CDS_heuristic(processing_times):
    jobs, machines = processing_times.shape
    m = machines-1
    johnson_proc_times = np.zeros((jobs,2))
    best_cost = np.inf
    best_seq = []
    for k in range(m):
        johnson_proc_times[:,0] += processing_times[:,k]
        johnson_proc_times[:,1] += processing_times[:,-k-1]
        seq = johnson_method(johnson_proc_times)
        cost = evaluate_sequence(seq,processing_times)
        if cost < best_cost:
            best_cost = cost
            best_seq = seq
    return best_seq, best_cost

In [ ]:
def sign(x):
    if x > 0:
        return 1
    elif x < 0:
        return -1
    else:
        return 0

In [ ]:
def min_gupta(job, processing_times):
    m = np.inf
    _, machines = processing_times.shape
    for i in range(machines-1):
        k = processing_times[job][i] + processing_times[job][i+1]
        if (k < m):
            m = k
    return m

In [ ]:
def gupta_heuristic(processing_times):
    jobs, machines = processing_times.shape
    f = []
    total_times = []
    for i in range(jobs):
        fi = sign(processing_times[i][0] - processing_times[i][machines-1]) / min_gupta(i,processing_times)
        f.append(fi)
        total_time = sum(processing_times[i])
        total_times.append(total_time)
    order = sorted(range(jobs), key=lambda k: (f[k], total_times[k]))
    return order

In [ ]:
def skewness(processing_times):
    jobs, machines = processing_times.shape
    skewnesses = []
    # Calculate the skewness for each job
    for i in range(jobs):
        avg = np.mean(processing_times[i,:])
        numerator = 0
        denominator = 0
        for j in range(machines):
            m = (processing_times[i,j] - avg)
            numerator += m**3
            denominator += m**2
        # Actually calculating the skewness
        numerator = numerator*(1/machines)
        denominator = (np.sqrt(denominator*(1/machines)))**3
        skewnesses.append(numerator/denominator)
    return np.array(skewnesses)

In [ ]:
def PRSKE_heuristic(processing_times):
    avg = np.mean(processing_times, axis=1)
    std = np.std(processing_times, axis=1, ddof=1)
    skw = skewness(processing_times)
    order = skw + std + avg
    sequence = [job for _ , job in sorted(zip(order, list(range(processing_times.shape[0]))),reverse=True)]
    return sequence, evaluate_sequence(sequence, processing_times)

# Tabu

In [ ]:
class TabuList:
    def __init__(self, max_size=7):
        self.max_size = max_size
        self.moves = []

    def add(self, move):
        if len(self.moves) >= self.max_size:
            self.moves.pop(0)
        self.moves.append(move)

    def is_tabu(self, move):
        return move in self.moves

In [ ]:

import random

def generate_neighbor(sequence,message):
    if (message == "random"):
        neighbor_type = random.choice([1, 2,3])
    elif (message == "swap"):
        neighbor_type = 1
    elif (message == "insert"):
        neighbor_type = 2
    elif (message == "insertblock"):
        neighbor_type = 3
    else:
        neighbor_type = random.choice([1, 2,3])

    if neighbor_type == 1:#swap
        return swap_neighbor(sequence)
    elif neighbor_type == 2:#insert 1 individual job
        return insert_neighbor(sequence)
    elif neighbor_type == 3:#insert a block of length k
        return block_insert_neighbor(sequence)

def swap_neighbor(sequence):
    # Select random positions i and j
    i, j = random.sample(range(len(sequence)), 2)
    neighbor = sequence[:]
    neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
    return neighbor

def insert_neighbor(sequence):
    # Select random positions i and j
    i, j = random.sample(range(len(sequence)), 2)
    neighbor = sequence[:]
    job_to_insert = neighbor.pop(i)
    neighbor.insert(j, job_to_insert)
    return neighbor

def block_insert_neighbor(sequence):
    # Select random positions i, j, and k
    i = random.randint(0, len(sequence) - 1)
    j = random.randint(0, len(sequence))
    k = random.randint(1, len(sequence) - i)
    neighbor = sequence[:]
    block_to_insert = neighbor[i:i+k]
    neighbor = neighbor[:j] + block_to_insert + neighbor[j:]
    return neighbor

def generate_candidate_list(sequence,neighbor_generation):
    n = len(sequence)
    candidate_list_size = 2 * n
    candidate_list = [generate_neighbor(sequence,neighbor_generation) for _ in range(candidate_list_size)]
    return candidate_list

def select_best_neighbor(processing_times,candidate_list, current_solution, tabu_list,best_makespan):
    best_neighbor = None

    # Iterate through the candidate list
    for neighbor in candidate_list:
        # Calculate makespan for the neighbor
        neighbor_makespan = evaluate_sequence(neighbor, processing_times)
        # Check if the neighbor is not tabu and improves the current solution
        if neighbor_makespan < best_makespan and not tabu_list.is_tabu(neighbor):
            best_neighbor = neighbor
            best_makespan = neighbor_makespan

    # If no improving move is found, examine the whole candidate list
    if best_neighbor is None:
        for neighbor in candidate_list:
            neighbor_makespan = evaluate_sequence(neighbor, processing_times)
            if neighbor_makespan < best_makespan:
                best_neighbor = neighbor
                best_makespan = neighbor_makespan

    return best_neighbor



def recherche_tabou(processing_times,s0,tabu_size, max_it, max_it_stagn,neighbor_generation):
    best_cmax = evaluate_sequence(s0, processing_times)
    best_solution = s0
    tabu_list = TabuList(tabu_size)
    stagnation_count = 0  # Counter to track stagnation iterations

    for i in range(max_it):
        candidate_list = generate_candidate_list(s0,neighbor_generation)
        best_neighbor = select_best_neighbor(processing_times, candidate_list, s0, tabu_list, best_cmax)
        # Check if there is no improving move
        if best_neighbor is None:
            stagnation_count += 1
            if stagnation_count >= max_it_stagn:
                break  # Terminate if stagnation persists
        else:
            stagnation_count = 0  # Reset stagnation counter
            # Update the current solution
            s0 = best_neighbor
            # Update the makespan of the best solution if a better solution is found
            current_makespan = evaluate_sequence(s0, processing_times)
            if current_makespan < best_cmax:
                best_solution = s0
                best_cmax = current_makespan
            # Update the tabu list
            tabu_list.add(s0)

    return best_solution, best_cmax

# GA

In [ ]:
import numpy as np
import math
import time
import random
import itertools
import queue
import pandas as pd
from IPython.display import display, Markdown

def initialization(Npop):
    population = []
    max_it_choices = {100, 200, 300, 500, 750, 900, 1200, 1500, 1750, 2000}
    max_it_stagn_choices={50,100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1400,1500,1600,1700,1800,1900,2000}
    for _ in range(Npop):
        # Generate random parameters for tabu search
        tabu_size = random.randint(3, 11)  # Example range for tabu list size
        max_it = random.choice(list(max_it_choices))  # Example range for maximum iterations
        max_it_stagn = random.choice(list(max_it_stagn_choices))  # Example range for maximum iterations of stagnation
        init_funct = random.randint(1,7)
        neighbor_generation = random.choice(["random", "swap", "insert", "insertblock"])  # Randomly select neighbor generation type
        individual = (tabu_size, max_it, max_it_stagn,init_funct, neighbor_generation)
        population.append(individual)
    return population


def calculateObj(processing_times,sol):
    # Run tabu search algorithm with specified parameters and return objective function value
    tabu_size, max_it, max_it_stagn,init_funct, neighbor_generation = sol
    if (init_funct ==1):
        seq, cmax  = neh_algorithm(processing_times)
    elif (init_funct ==2):
        seq, cmax = ham_heuristic(processing_times)
    elif (init_funct ==3):
        seq = palmer_heuristic(processing_times)
        cmax = evaluate_sequence(seq, processing_times)
    elif (init_funct ==4):
        seq , cmax = CDS_heuristic(processing_times)
    elif (init_funct ==5):
        seq = gupta_heuristic(processing_times)
        cmax = evaluate_sequence(seq, processing_times)
    elif (init_funct ==6):
        seq, cmax = PRSKE_heuristic(processing_times)
    else :
        transposing_times = np.transpose(processing_times)
        num_jobs = len(transposing_times[0])
        seq = list(range(num_jobs))
        random.shuffle(seq)
        cmax = evaluate_sequence(seq, processing_times)
    best_seq, best_cmax = recherche_tabou(processing_times,seq,tabu_size, max_it, max_it_stagn,neighbor_generation)

    return best_cmax

def selection(pop,processing_times):
    popObj = []
    for i in range(len(pop)):
        popObj.append([calculateObj(processing_times,pop[i]), i])

    popObj.sort()

    distr = []
    distrInd = []

    for i in range(len(pop)):
        distrInd.append(popObj[i][1])
        prob = (2*(i+1)) / (len(pop) * (len(pop)+1))
        distr.append(prob)

    parents = []
    for i in range(len(pop)):
        parents.append(list(np.random.choice(distrInd, 2, p=distr)))

    return parents

def crossover(parent1, parent2):
    # Perform crossover by combining parameters of two parents
    crossover_point = random.randint(0, len(parent1) - 1)
    child = parent1[:crossover_point] + parent2[crossover_point:]
    return child

def mutation(individual):
    # Perform mutation by randomly changing one parameter
    mutation_point = random.randint(0, len(individual) - 1)
    mutated_value = None
    max_it_choices = {100, 200, 300, 500, 750, 900, 1200, 1500, 1750, 2000}
    max_it_stagn_choices={50,100,200,300,400,500,600,700,800,900,1000,1100,1200,1300,1400,1500,1600,1700,1800,1900,2000}
    if mutation_point == 0:
        mutated_value = random.randint(3, 20)  # Example range for tabu list size
    elif mutation_point == 1:
        mutated_value = random.choice(list(max_it_choices))   # Example range for maximum iterations
    elif mutation_point == 2:
        mutated_value = random.choice(list(max_it_stagn_choices))   # Example range for maximum iterations of stagnation
    elif mutation_point == 3:
        mutated_value = random.choice(["random", "swap", "insert", "insertblock"]) # Randomly select neighbor generation type
    elif mutation_point == 4:
        mutated_value = random.randint(1, 7)
    individual = list(individual)
    individual[mutation_point] = mutated_value
    return tuple(individual)


def elitistUpdate(oldPop, newPop, num_elites, processing_times):
    # Combine old and new populations
    combined_pop = oldPop + newPop

    # Sort the combined population based on objective function values
    sorted_pop = sorted(combined_pop, key=lambda x: calculateObj(processing_times, x))

    # Select the elite individuals from the sorted population
    elite_individuals = sorted_pop[:num_elites]

    # Update the new population with elite individuals
    newPop[:num_elites] = elite_individuals

    return newPop

def findBestSolution(pop, processing_times):
    # Calculate objective values for all individuals and track the best one
    popObj = []
    bestObj = float('inf')
    bestInd = -1
    totalObj = 0

    for i in range(len(pop)):
        tObj = calculateObj(processing_times, pop[i])
        popObj.append(tObj)
        totalObj += tObj
        if tObj < bestObj:
            bestObj = tObj
            bestInd = i

    avgObj = totalObj / len(pop)
    return bestInd, bestObj, avgObj


# Selection methods

In [ ]:
def roulette_wheel_selection(pop, processing_times):
    # Calculate fitnesses (inverse of objective values assuming we minimize the objective)
    fitnesses = [1 / calculateObj(processing_times, individual) for individual in pop]
    total_fitness = sum(fitnesses)
    probabilities = [fitness / total_fitness for fitness in fitnesses]

    # Select 50% of the population
    num_to_select = len(pop) // 2
    selected_indices = np.random.choice(len(pop), size=num_to_select, p=probabilities, replace=True)

    # Ensure there are no repeats by shuffling and creating pairs
    random.shuffle(selected_indices)
    parents = [(selected_indices[i], selected_indices[(i + 1) % num_to_select]) for i in range(num_to_select)]
    return parents


def rank_based_selection(pop, processing_times):
    # Calculate objective values for all individuals
    popObj = [calculateObj(processing_times, individual) for individual in pop]

    # Sort individuals based on their objective values
    sorted_indices = sorted(range(len(pop)), key=lambda i: popObj[i])

    # Assign ranks
    ranks = [0] * len(pop)
    for rank, idx in enumerate(sorted_indices, start=1):
        ranks[idx] = rank

    # Calculate total rank sum
    total_rank_sum = sum(ranks)

    # Calculate selection probabilities based on ranks
    probabilities = [rank / total_rank_sum for rank in ranks]

    # Select 50% of the population without replacement
    num_to_select = len(pop) // 2
    selected_indices = np.random.choice(len(pop), size=num_to_select, p=probabilities, replace=False)

    # Ensure there are no repeats by shuffling and creating pairs
    random.shuffle(selected_indices)

    # Ensure even number of pairs
    if len(selected_indices) % 2 != 0:
        selected_indices = selected_indices[:-1]

    parents = [(selected_indices[i], selected_indices[i + 1]) for i in range(0, len(selected_indices), 2)]
    return parents

def elitist_selection(pop, processing_times, num_to_select):
    # Calculate objective values for all individuals
    popObj = [calculateObj(processing_times, individual) for individual in pop]

    # Sort individuals based on their objective values
    sorted_indices = sorted(range(len(pop)), key=lambda i: popObj[i])

    # Select the top num_to_select individuals
    selected_indices = sorted_indices[:num_to_select]

    # Ensure there are no repeats by shuffling and creating pairs
    random.shuffle(selected_indices)

    # Handle odd number of selections by duplicating the first element
    if num_to_select % 2 == 1:
        selected_indices.append(selected_indices[0])

    # Create pairs of parents from the selected indices
    parents = [(selected_indices[i], selected_indices[i + 1]) for i in range(0, num_to_select, 2)]
    return parents


def tournament_selection(pop, processing_times, num_to_select, k=2, p=0.75):
    def tournament(participants):
        # Select the best individual from the tournament with probability p
        participants.sort(key=lambda x: calculateObj(processing_times, x[1]))
        if random.random() < p:
            return participants[0][0]
        else:
            return random.choice(participants[1:])[0]

    selected_indices = []
    while len(selected_indices) < num_to_select:
        # Randomly select k individuals for the tournament
        tournament_participants = random.sample(list(enumerate(pop)), k)
        winner_index = tournament(tournament_participants)
        if winner_index not in selected_indices:
            selected_indices.append(winner_index)

    # Ensure there are no repeats by shuffling and creating pairs
    random.shuffle(selected_indices)

    # Handle odd number of selections by duplicating the first element
    if num_to_select % 2 == 1:
        selected_indices.append(selected_indices[0])

    parents = [(selected_indices[i], selected_indices[i + 1]) for i in range(0, num_to_select, 2)]
    return parents

def combined_replacement_strategy(oldPop, newPop, num_elites, processing_times):
    combined_pop = oldPop + newPop
    combined_pop.sort(key=lambda x: calculateObj(processing_times, x))
    new_population = combined_pop[:num_elites]
    remaining_indices = list(range(num_elites, len(combined_pop)))
    random.shuffle(remaining_indices)
    while len(new_population) < len(oldPop):
        new_population.append(combined_pop[remaining_indices.pop()])
    return new_population

Npop = 10
generations = 5
num_elites = 6
matrix = np.array([
    [77, 94, 9, 57, 29, 79, 55, 73, 65, 86, 25, 39, 76, 24, 38, 5, 91, 29, 22, 27],
    [39, 31, 46, 18, 93, 58, 85, 58, 97, 10, 79, 93, 2, 87, 17, 18, 10, 50, 8, 26],
    [14, 21, 15, 10, 85, 46, 42, 18, 36, 2, 44, 89, 6, 3, 1, 43, 81, 57, 76, 59],
    [11, 2, 36, 30, 89, 10, 88, 22, 31, 9, 43, 91, 26, 3, 75, 99, 63, 83, 70, 84],
    [83, 13, 84, 46, 20, 33, 74, 42, 33, 71, 32, 48, 42, 99, 7, 54, 8, 73, 30, 75]
])
def HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times, k=2, p=0.75):
    pop = initialization(Npop)
    for _ in range(generations):
        parents = tournament_selection(pop, processing_times, Npop // 2, k, p)
        new_pop = []
        for parent1_idx, parent2_idx in parents:
            parent1 = pop[parent1_idx]
            parent2 = pop[parent2_idx]
            if random.random() < 0.9:  # Assume 90% crossover probability
                child1 = crossover(parent1, parent2)
                child2 = crossover(parent2, parent1)
            else:
                child1, child2 = parent1, parent2
            if random.random() < 0.5:
                new_pop.append(mutation(child1))
                new_pop.append(mutation(child2))
            else:
                new_pop.append(child1)
                new_pop.append(child2)
        pop = combined_replacement_strategy(pop, new_pop, num_elites, processing_times)
    best_ind, best_obj, avg_obj = findBestSolution(pop, processing_times)

    return pop[best_ind], best_obj, avg_obj
processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
#derivation = deviation(Cmax, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("derivation", )

In [ ]:
Npop = 10
generations = 10
num_elites = 7
matrix = np.array([
    [34, 20, 57, 47, 62, 40, 74, 94,  9, 62, 86, 13, 78, 46, 83, 52, 13, 70, 40, 60],
    [ 5, 48, 80, 43, 34,  2, 87, 68, 28, 84, 30, 35, 42, 39, 85, 34, 36,  9, 96, 84],
    [86, 35,  5, 93, 74, 12, 40, 95, 80,  6, 92, 14, 83, 49, 36, 38, 43, 89, 94, 33],
    [28, 39, 55, 21, 25, 88, 59, 40, 90, 18, 33, 10, 59, 92, 15, 77, 31, 85, 85, 99],
    [ 8, 91, 45, 55, 75, 18, 59, 86, 45, 89, 11, 54, 38, 41, 64, 98, 83, 36, 61, 19]
])
def HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times, k=2, p=0.75):
    pop = initialization(Npop)
    for _ in range(generations):
        parents = tournament_selection(pop, processing_times, Npop // 2, k, p)
        new_pop = []
        for parent1_idx, parent2_idx in parents:
            parent1 = pop[parent1_idx]
            parent2 = pop[parent2_idx]
            if random.random() < 0.9:  # Assume 90% crossover probability
                child1 = crossover(parent1, parent2)
                child2 = crossover(parent2, parent1)
            else:
                child1, child2 = parent1, parent2
            if random.random() < 0.5:
                new_pop.append(mutation(child1))
                new_pop.append(mutation(child2))
            else:
                new_pop.append(child1)
                new_pop.append(child2)
        pop = combined_replacement_strategy(Pop, new_Pop, num_elites, processing_times)
    best_ind, best_obj, avg_obj = findBestSolution(pop, processing_times)
    return pop[best_ind], best_obj, avg_obj
processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
#derivation = deviation(Cmax, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("derivation", )

In [ ]:
Npop = 10
generations = 5
num_elites = 6
matrix = np.array([
    [34, 20, 57, 47, 62, 40, 74, 94,  9, 62, 86, 13, 78, 46, 83, 52, 13, 70, 40, 60],
    [ 5, 48, 80, 43, 34,  2, 87, 68, 28, 84, 30, 35, 42, 39, 85, 34, 36,  9, 96, 84],
    [86, 35,  5, 93, 74, 12, 40, 95, 80,  6, 92, 14, 83, 49, 36, 38, 43, 89, 94, 33],
    [28, 39, 55, 21, 25, 88, 59, 40, 90, 18, 33, 10, 59, 92, 15, 77, 31, 85, 85, 99],
    [ 8, 91, 45, 55, 75, 18, 59, 86, 45, 89, 11, 54, 38, 41, 64, 98, 83, 36, 61, 19]
])
def HHGA_with_roulette_wheel_selection(Npop, generations, num_elites, processing_times):
    pop = initialization(Npop)
    for _ in range(generations):
        parents = roulette_wheel_selection(pop, processing_times)
        new_pop = []
        for parent1_idx, parent2_idx in parents:
            parent1 = pop[parent1_idx]
            parent2 = pop[parent2_idx]
            if random.random() < 0.9:  # Assume 90% crossover probability
                child1 = crossover(parent1, parent2)
                child2 = crossover(parent2, parent1)
            else:
                child1, child2 = parent1, parent2
            if random.random() < 0.5:
                new_pop.append(mutation(child1))
                new_pop.append(mutation(child2))
            else:
                new_pop.append(child1)
                new_pop.append(child2)
        pop = combined_replacement_strategy(Pop, new_Pop, num_elites, processing_times)
    best_ind, best_obj, avg_obj = findBestSolution(pop, processing_times)
    return pop[best_ind], best_obj, avg_obj
processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_roulette_wheel_selection(Npop, generations, num_elites, processing_times)
elapsed_time = time.time() - start_time
#derivation = deviation(Cmax, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("derivation", )

In [ ]:
Npop = 10
generations = 5
num_elites = 6
matrix = np.array([
    [34, 20, 57, 47, 62, 40, 74, 94,  9, 62, 86, 13, 78, 46, 83, 52, 13, 70, 40, 60],
    [ 5, 48, 80, 43, 34,  2, 87, 68, 28, 84, 30, 35, 42, 39, 85, 34, 36,  9, 96, 84],
    [86, 35,  5, 93, 74, 12, 40, 95, 80,  6, 92, 14, 83, 49, 36, 38, 43, 89, 94, 33],
    [28, 39, 55, 21, 25, 88, 59, 40, 90, 18, 33, 10, 59, 92, 15, 77, 31, 85, 85, 99],
    [ 8, 91, 45, 55, 75, 18, 59, 86, 45, 89, 11, 54, 38, 41, 64, 98, 83, 36, 61, 19]
])
def HHGA_with_rank_selection(Npop, generations, num_elites, processing_times):
    pop = initialization(Npop)
    for _ in range(generations):
        parents = rank_based_selection(pop, processing_times)
        new_pop = []
        for parent1_idx, parent2_idx in parents:
            parent1 = pop[parent1_idx]
            parent2 = pop[parent2_idx]
            if random.random() < 0.9:  # Assume 90% crossover probability
                child1 = crossover(parent1, parent2)
                child2 = crossover(parent2, parent1)
            else:
                child1, child2 = parent1, parent2
            if random.random() < 0.5:
                new_pop.append(mutation(child1))
                new_pop.append(mutation(child2))
            else:
                new_pop.append(child1)
                new_pop.append(child2)
        pop = combined_replacement_strategy(Pop, new_Pop, num_elites, processing_times)
    best_ind, best_obj, avg_obj = findBestSolution(pop, processing_times)
    return pop[best_ind], best_obj, avg_obj
processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_rank_selection(Npop, generations, num_elites, processing_times)
elapsed_time = time.time() - start_time
#derivation = deviation(Cmax, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("derivation", )

In [ ]:
Npop = 10
generations = 5
num_elites = 6
matrix = np.array([
    [34, 20, 57, 47, 62, 40, 74, 94,  9, 62, 86, 13, 78, 46, 83, 52, 13, 70, 40, 60],
    [ 5, 48, 80, 43, 34,  2, 87, 68, 28, 84, 30, 35, 42, 39, 85, 34, 36,  9, 96, 84],
    [86, 35,  5, 93, 74, 12, 40, 95, 80,  6, 92, 14, 83, 49, 36, 38, 43, 89, 94, 33],
    [28, 39, 55, 21, 25, 88, 59, 40, 90, 18, 33, 10, 59, 92, 15, 77, 31, 85, 85, 99],
    [ 8, 91, 45, 55, 75, 18, 59, 86, 45, 89, 11, 54, 38, 41, 64, 98, 83, 36, 61, 19]
])
def HHGA_with_elitist_selection(Npop, generations, num_elites, processing_times):
    pop = initialization(Npop)
    for _ in range(generations):
        parents = elitist_selection(pop, processing_times, num_elites)
        new_pop = []
        for parent1_idx, parent2_idx in parents:
            parent1 = pop[parent1_idx]
            parent2 = pop[parent2_idx]
            if random.random() < 0.9:  # Assume 90% crossover probability
                child1 = crossover(parent1, parent2)
                child2 = crossover(parent2, parent1)
            else:
                child1, child2 = parent1, parent2
            if random.random() < 0.5:
                new_pop.append(mutation(child1))
                new_pop.append(mutation(child2))
            else:
                new_pop.append(child1)
                new_pop.append(child2)
        pop = combined_replacement_strategy(Pop, new_Pop, num_elites, processing_times)
    best_ind, best_obj, avg_obj = findBestSolution(pop, processing_times)
    return pop[best_ind], best_obj, avg_obj
processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_elitist_selection(Npop, generations, num_elites, processing_times)
elapsed_time = time.time() - start_time
#derivation = deviation(Cmax, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("derivation", )

# Tests on taillard instances :

In [ ]:
UP =  1206
Npop = 10
generations = 5
num_elites = 4
matrix = np.array([
    [34, 20, 57, 47, 62, 40, 74, 94,  9, 62, 86, 13, 78, 46, 83, 52, 13, 70, 40, 60],
    [ 5, 48, 80, 43, 34,  2, 87, 68, 28, 84, 30, 35, 42, 39, 85, 34, 36,  9, 96, 84],
    [86, 35,  5, 93, 74, 12, 40, 95, 80,  6, 92, 14, 83, 49, 36, 38, 43, 89, 94, 33],
    [28, 39, 55, 21, 25, 88, 59, 40, 90, 18, 33, 10, 59, 92, 15, 77, 31, 85, 85, 99],
    [ 8, 91, 45, 55, 75, 18, 59, 86, 45, 89, 11, 54, 38, 41, 64, 98, 83, 36, 61, 19]
])
processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
dev = deviation(best_obj, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("deviation", )

In [ ]:
UP =  1278
Npop = 10
generations = 5
num_elites = 4
matrix = np.array([
    [54, 83, 15, 71, 77, 36, 53, 38, 27, 87, 76, 91, 14, 29, 12, 77, 32, 87, 68, 94],
    [79,  3, 11, 99, 56, 70, 99, 60,  5, 56,  3, 61, 73, 75, 47, 14, 21, 86,  5, 77],
    [16, 89, 49, 15, 89, 45, 60, 23, 57, 64,  7,  1, 63, 41, 63, 47, 26, 75, 77, 40],
    [66, 58, 31, 68, 78, 91, 13, 59, 49, 85, 85,  9, 39, 41, 56, 40, 54, 77, 51, 31],
    [58, 56, 20, 85, 53, 35, 53, 41, 69, 13, 86, 72,  8, 49, 47, 87, 58, 18, 68, 28]
])

processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
dev = deviation(best_obj, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("deviation", )

In [ ]:
UP =  1239
Npop = 10
generations = 5
num_elites = 4
matrix = np.array([
    [15, 64, 64, 48,  9, 91, 27, 34, 42,  3, 11, 54, 27, 30,  9, 15, 88, 55, 50, 57],
    [28,  4, 43, 93,  1, 81, 77, 69, 52, 28, 28, 77, 42, 53, 46, 49, 15, 43, 65, 41],
    [77, 36, 57, 15, 81, 82, 98, 97, 12, 35, 84, 70, 27, 37, 59, 42, 57, 16, 11, 34],
    [ 1, 59, 95, 49, 90, 78,  3, 69, 99, 41, 73, 28, 99, 13, 59, 47,  8, 92, 87, 62],
    [45, 73, 59, 63, 54, 98, 39, 75, 33,  8, 86, 41, 41, 22, 43, 34, 80, 16, 37, 94]
])

processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
dev = deviation(best_obj, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("deviation", )

In [ ]:
UP =  1293
Npop = 10
generations = 5
num_elites = 4
matrix = np.array([
    [53, 19, 99, 62, 88, 93, 34, 72, 42, 65, 39, 79,  9, 26, 72, 29, 36, 48, 57, 95],
    [93, 79, 88, 77, 94, 39, 74, 46, 17, 30, 62, 77, 43, 98, 48, 14, 45, 25, 98, 30],
    [90, 92, 35, 13, 75, 55, 80, 67,  3, 93, 54, 67, 25, 77, 38, 98, 96, 20, 15, 36],
    [65, 97, 27, 25, 61, 24, 97, 61, 75, 92, 73, 21, 29,  3, 96, 51, 26, 44, 56, 31],
    [64, 38, 44, 46, 66, 31, 48, 27, 82, 51, 90, 63, 85, 36, 69, 67, 81, 18, 81, 72]
])

processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
dev = deviation(best_obj, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("deviation", )

In [ ]:
UP =  1582
Npop = 10
generations = 5
num_elites = 4
matrix = np.array([
    [74, 21, 58,  4, 21, 28, 58, 83, 31, 61, 94, 44, 97, 94, 66,  6, 37, 22, 99, 83],
    [28,  3, 27, 61, 34, 76, 64, 87, 54, 98, 76, 41, 70, 43, 42, 79, 88, 15, 49, 72],
    [89, 52, 56, 13,  7, 32, 32, 98, 46, 60, 23, 87,  7, 36, 26, 85,  7, 34, 36, 48],
    [60, 88, 26, 58, 76, 98, 29, 47, 79, 26, 19, 48, 95, 78, 77, 90, 24, 10, 85, 55],
    [54, 66, 12, 57, 70, 82, 99, 84, 16, 41, 23, 11, 68, 58, 30,  5,  5, 39, 58, 31],
    [92, 11, 54, 97, 57, 53, 65, 77, 51, 36, 53, 19, 54, 86, 40, 56, 79, 74, 24,  3],
    [ 9,  8, 88, 72, 27, 22, 50,  2, 49, 82, 93, 96, 43, 13, 60, 11, 37, 91, 84, 67],
    [ 4, 18, 25, 28, 95, 51, 84, 18,  6, 90, 69, 61, 57,  5, 75,  4, 38, 28,  4, 80],
    [25, 15, 91, 49, 56, 10, 62, 70, 76, 99, 58, 83, 84, 64, 74, 14, 18, 48, 96, 86],
    [15, 84,  8, 30, 95, 79,  9, 91, 76, 26, 42, 66, 70, 91, 67,  3, 98,  4, 71, 62]
])


processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
dev = deviation(best_obj, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("deviation", )

In [ ]:
UP = 1378
Npop = 10
generations = 5
num_elites = 4
matrix = np.array([
    [94, 43,  6, 47, 45, 51, 73, 49, 31, 58, 19, 36, 54, 75,  7,  5, 82, 20, 31, 32],
    [ 3, 18, 43, 41, 83, 62, 27, 75, 52, 58, 18,  1, 61, 67, 55, 72, 16, 12, 87, 21],
    [39, 23, 28, 31, 86, 19, 85, 90, 77,  4, 85, 81,  9, 52, 67, 77, 45,  7, 21, 26],
    [ 1, 92, 83, 43,  3,  3, 51,  5, 38, 37, 44, 66, 31,  8, 79, 42, 85, 92, 89, 29],
    [63, 96, 50, 12, 15, 11, 33,  7,  4, 58, 27,  7,  7, 54, 31, 94, 25,  8, 61, 51],
    [86, 36, 19, 71,  8, 77,  8,  6, 40, 39, 24, 82, 69, 82, 39, 52, 85, 48, 22, 57],
    [44, 25, 85, 86, 73, 58, 95, 13, 50, 39, 24, 55, 58, 79, 42, 98, 44, 16, 13, 74],
    [19, 82, 12, 70,  6, 64,  3,  4, 29, 43, 40, 77, 88, 74, 13, 13, 17, 45,  2, 22],
    [55, 46, 68, 15, 55, 74, 42, 40, 88, 68, 67, 67,  8, 13, 74, 47,  3, 95, 36, 46],
    [67, 66, 66, 32,  8, 30, 92, 40, 13, 79, 19, 29, 27, 18, 42, 86, 30, 41, 27, 50]
])

processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
dev = deviation(best_obj, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("deviation", )

In [ ]:
UP =  1397
Npop = 10
generations = 5
num_elites = 4
matrix = np.array([
    [47, 77, 66, 13, 77, 20, 29, 11, 85, 98, 36, 92, 99, 65, 34, 35, 42, 28, 28,  5],
    [80, 46, 37, 86, 85, 67, 22, 67, 20,  1, 39, 25,  1, 41, 74, 39,  2, 20,  3, 29],
    [51, 79, 35, 63, 18, 67, 73,  2, 23, 59, 46, 12, 17, 87, 35, 27, 71, 11, 97, 83],
    [63, 22, 47, 46, 72, 16, 35,  2, 22, 50, 58, 46, 19, 82, 69, 14, 94, 39, 62, 27],
    [60, 20, 13, 64, 67, 25, 39, 40, 72, 28, 36, 60,  2,  8, 33, 85, 51,  6,  5, 72],
    [70, 96, 56, 22, 44, 83, 66, 56, 27, 52, 46, 83, 76, 25, 28, 26, 98, 64, 66, 15],
    [39, 75, 66, 89, 56, 42, 90, 77, 76, 43, 14,  3, 57, 98, 47, 35, 58,  6, 95,  9],
    [33,  1, 59, 85,  1, 85, 65, 47, 13, 12, 23, 21, 89, 51, 34, 62,  6, 22, 75, 22],
    [ 5, 37, 72,  6, 90, 71, 47, 60, 93, 17, 65, 12, 97, 29, 41,  3, 46, 33, 55, 10],
    [ 9, 14, 37, 54, 14, 53, 27, 64, 25, 79, 30, 33,  5, 96,  1, 51, 42, 78, 70, 91]
])

processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
dev = deviation(best_obj, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("deviation", )

In [ ]:
UP =  1397
Npop = 10
generations = 5
num_elites = 4
matrix = np.array([
    [47, 77, 66, 13, 77, 20, 29, 11, 85, 98, 36, 92, 99, 65, 34, 35, 42, 28, 28,  5],
    [80, 46, 37, 86, 85, 67, 22, 67, 20,  1, 39, 25,  1, 41, 74, 39,  2, 20,  3, 29],
    [51, 79, 35, 63, 18, 67, 73,  2, 23, 59, 46, 12, 17, 87, 35, 27, 71, 11, 97, 83],
    [63, 22, 47, 46, 72, 16, 35,  2, 22, 50, 58, 46, 19, 82, 69, 14, 94, 39, 62, 27],
    [60, 20, 13, 64, 67, 25, 39, 40, 72, 28, 36, 60,  2,  8, 33, 85, 51,  6,  5, 72],
    [70, 96, 56, 22, 44, 83, 66, 56, 27, 52, 46, 83, 76, 25, 28, 26, 98, 64, 66, 15],
    [39, 75, 66, 89, 56, 42, 90, 77, 76, 43, 14,  3, 57, 98, 47, 35, 58,  6, 95,  9],
    [33,  1, 59, 85,  1, 85, 65, 47, 13, 12, 23, 21, 89, 51, 34, 62,  6, 22, 75, 22],
    [ 5, 37, 72,  6, 90, 71, 47, 60, 93, 17, 65, 12, 97, 29, 41,  3, 46, 33, 55, 10],
    [ 9, 14, 37, 54, 14, 53, 27, 64, 25, 79, 30, 33,  5, 96,  1, 51, 42, 78, 70, 91]
])

processing_times = np.transpose(matrix)
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_tournament_selection(Npop, generations, num_elites, processing_times,k=2,p=0.7)
elapsed_time = time.time() - start_time
dev = deviation(best_obj, UP)
print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("elapsed time :", elapsed_time)
print("deviation", )

In [ ]:
!pip install 'shimmy>=0.2.1'


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 9.0 MB/s eta 0:00:00


In [ ]:
!pip install stable-baselines3 gymnasium shimmy numpy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 3.3 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl (166.0 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-many

In [ ]:
!pip install stable-baselines3 gym numpy


In [ ]:
!pip install stable-baselines3 gymnasium shimmy numpy


In [ ]:
import numpy as np
import random
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv





import gym
from gym import spaces
import numpy as np

class GAEnv(gym.Env):
    def __init__(self, processing_times):
        super(GAEnv, self).__init__()
        self.processing_times = processing_times
        self.max_tabu_size = 20
        self.max_iterations = 2000
        self.max_stagnation = 2000

        self.action_space = spaces.Discrete(5)
        self.observation_space = spaces.Box(low=0, high=1, shape=(5,), dtype=np.float32)

        self.current_solution = None
        self.current_step = 0
        self.reset()

    def step(self, action):
        done = False
        reward = -self.calculateObj(self.processing_times, self.current_solution)
        self.current_step += 1

        if self.current_step >= self.max_iterations:
            done = True

        self.current_solution = self.random_solution()
        return self.get_observation(), reward, done, {}

    def reset(self):
        self.current_solution = self.random_solution()
        self.current_step = 0
        return self.get_observation()

    def get_observation(self):
        tabu_size, max_it, max_it_stagn, init_funct, neighbor_generation = self.current_solution
        return np.array([tabu_size / self.max_tabu_size, max_it / self.max_iterations, max_it_stagn / self.max_stagnation, init_funct / 7, neighbor_generation / 4], dtype=np.float32)

    def random_solution(self):
        tabu_size = random.randint(3, self.max_tabu_size)
        max_it = random.randint(100, self.max_iterations)
        max_it_stagn = random.randint(50, self.max_stagnation)
        init_funct = random.randint(1, 7)
        neighbor_generation = random.randint(0, 3)
        return (tabu_size, max_it, max_it_stagn, init_funct, neighbor_generation)

    def calculateObj(self, processing_times, sol):
        # Replace this with your actual objective calculation code
        return random.random()

import numpy as np
import random
import time

def initialization_with_rl(Npop, model, env):
    population = []
    for _ in range(Npop):
        obs = env.reset()
        action, _ = model.predict(obs, deterministic=True)
        action = np.clip(action, 0, env.action_space.n - 1).astype(int)

        tabu_size = random.randint(3, env.envs[0].max_tabu_size)
        max_it = random.randint(100, env.envs[0].max_iterations)
        max_it_stagn = random.randint(50, env.envs[0].max_stagnation)
        init_funct = action  # Use action directly for initialization function
        neighbor_generation = random.randint(0, 3)

        population.append((tabu_size, max_it, max_it_stagn, init_funct, neighbor_generation))
    return population



def calculateCmax(processing_times, sol):
    tabu_size, max_it, max_it_stagn, init_funct, neighbor_generation = sol
    if init_funct == 1:
        seq, cmax = neh_algorithm(processing_times)
    elif init_funct == 2:
        seq, cmax = ham_heuristic(processing_times)
    elif init_funct == 3:
        seq = palmer_heuristic(processing_times)
        cmax = evaluate_sequence(seq, processing_times)
    elif init_funct == 4:
        seq, cmax = CDS_heuristic(processing_times)
    elif init_funct == 5:
        seq = gupta_heuristic(processing_times)
        cmax = evaluate_sequence(seq, processing_times)
    elif init_funct == 6:
        seq, cmax = PRSKE_heuristic(processing_times)
    else:
        transposing_times = np.transpose(processing_times)
        num_jobs = len(transposing_times[0])
        seq = list(range(num_jobs))
        random.shuffle(seq)
        cmax = evaluate_sequence(seq, processing_times)
    best_seq, best_cmax = recherche_tabou(processing_times, seq, tabu_size, max_it, max_it_stagn, neighbor_generation)
    return best_seq, best_cmax



def calculateObj(processing_times, sol):
    tabu_size, max_it, max_it_stagn, init_funct, neighbor_generation = sol
    if init_funct == [1]:
        seq, cmax = neh_algorithm(processing_times)
    elif init_funct == [2]:
        seq, cmax = ham_heuristic(processing_times)
    elif init_funct == [3]:
        seq = palmer_heuristic(processing_times)
        cmax = evaluate_sequence(seq, processing_times)
    elif init_funct == [4]:
        seq, cmax = CDS_heuristic(processing_times)
    elif init_funct == [5]:
        seq = gupta_heuristic(processing_times)
        cmax = evaluate_sequence(seq, processing_times)
    elif init_funct == [6]:
        seq, cmax = PRSKE_heuristic(processing_times)
    else:
        transposing_times = np.transpose(processing_times)
        num_jobs = len(transposing_times[0])
        seq = list(range(num_jobs))
        random.shuffle(seq)
        cmax = evaluate_sequence(seq, processing_times)
    best_seq, best_cmax = recherche_tabou(processing_times, seq, tabu_size, max_it, max_it_stagn, neighbor_generation)
    return best_cmax

def selection(pop, processing_times):
    popObj = []
    for i in range(len(pop)):
        popObj.append([calculateObj(processing_times, pop[i]), i])
    popObj.sort()
    distr = []
    distrInd = []
    for i in range(len(pop)):
        distrInd.append(popObj[i][1])
        prob = (2 * (i + 1)) / (len(pop) * (len(pop) + 1))
        distr.append(prob)
    parents = []
    for i in range(len(pop)):
        parents.append(list(np.random.choice(distrInd, 2, p=distr)))
    return parents

def crossover(parent1, parent2):
    crossover_point = random.randint(0, len(parent1) - 1)
    child = parent1[:crossover_point] + parent2[crossover_point:]
    return child

def mutation(individual):
    mutation_point = random.randint(0, len(individual) - 1)
    mutated_value = None
    if mutation_point == 0:
        mutated_value = random.randint(3, 20)
    elif mutation_point == 1:
        mutated_value = random.randint(100, 2000)
    elif mutation_point == 2:
        mutated_value = random.randint(50, 2000)
    elif mutation_point == 3:
        mutated_value = random.randint(1, 7)
    elif mutation_point == 4:
        mutated_value = random.randint(0, 3)
    individual = list(individual)
    individual[mutation_point] = mutated_value
    return tuple(individual)

def elitistUpdate(oldPop, newPop, num_elites, processing_times):
    combined_pop = oldPop + newPop
    sorted_pop = sorted(combined_pop, key=lambda x: calculateObj(processing_times, x))
    elite_individuals = sorted_pop[:num_elites]
    newPop[:num_elites] = elite_individuals
    return newPop

def findBestSolution(pop, processing_times):
    bestObj = calculateObj(processing_times, pop[0])
    avgObj = bestObj
    bestInd = 0
    for i in range(1, len(pop)):
        tObj = calculateObj(processing_times, pop[i])
        avgObj = avgObj + tObj
        if tObj < bestObj:
            bestObj = tObj
            bestInd = i
    return bestInd, bestObj, avgObj / len(pop)


def HHGA_with_RL(Npop, generations, num_elites, processing_times, model, env):
    population = initialization_with_rl(Npop, model, env)
    best_solution = None
    best_obj = float('inf')
    avg_obj = []

    for generation in range(generations):
        fitness = []
        for sol in population:
            obj_val = env.envs[0].calculateObj(processing_times, sol)
            fitness.append((sol, obj_val))
            if obj_val < best_obj:
                best_obj = obj_val
                best_solution = sol

        fitness.sort(key=lambda x: x[1])
        elites = [sol for sol, _ in fitness[:num_elites]]
        avg_obj.append(np.mean([f for _, f in fitness]))

        new_population = elites[:]
        while len(new_population) < Npop:
            parent1, parent2 = random.sample(elites, 2)
            child = crossover(parent1, parent2)
            child = mutation(child)
            new_population.append(child)


        population = new_population

    tabu_size, max_it, max_it_stagn, init_funct, neighbor_generation = best_solution
    if init_funct == [1]:
        seq, cmax = neh_algorithm(processing_times)
    elif init_funct == [2]:
        seq, cmax = ham_heuristic(processing_times)
    elif init_funct == [3]:
        seq = palmer_heuristic(processing_times)
        cmax = evaluate_sequence(seq, processing_times)
    elif init_funct == [4]:
        seq, cmax = CDS_heuristic(processing_times)
    elif init_funct == [5]:
        seq = gupta_heuristic(processing_times)
        cmax = evaluate_sequence(seq, processing_times)
    elif init_funct == [6]:
        seq, cmax = PRSKE_heuristic(processing_times)
    else:
        transposing_times = np.transpose(processing_times)
        num_jobs = len(transposing_times[0])
        seq = list(range(num_jobs))
        random.shuffle(seq)
        cmax = evaluate_sequence(seq, processing_times)
    best_seq, best_cmax = recherche_tabou(processing_times, seq, tabu_size, max_it, max_it_stagn, neighbor_generation)


    return best_solution, best_obj, avg_obj,best_seq, best_cmax


In [ ]:
# Define the processing times matrix
processing_times = np.array([
    [46, 52, 79, 45, 97, 10, 44, 24, 85, 75, 66, 49, 95, 61, 19, 47, 84, 13, 11, 19, 98, 2, 85, 44, 7, 73, 19, 69, 12, 73, 85, 23, 53, 16, 88, 8, 26, 42, 58, 63, 7, 2, 44, 38, 24, 76, 85, 61, 32, 90],
    [61, 87, 51, 25, 73, 93, 28, 90, 94, 59, 64, 2, 16, 35, 53, 40, 81, 26, 85, 4, 4, 10, 63, 96, 55, 71, 66, 94, 7, 15, 11, 99, 37, 50, 56, 69, 22, 56, 67, 63, 96, 74, 4, 42, 40, 30, 93, 36, 25, 87],
    [3, 1, 58, 85, 33, 71, 58, 56, 64, 43, 48, 69, 96, 35, 82, 53, 64, 11, 61, 36, 53, 87, 88, 10, 32, 38, 25, 24, 90, 7, 11, 49, 2, 76, 17, 32, 39, 9, 83, 69, 67, 28, 88, 23, 91, 71, 3, 26, 41, 96],
    [51, 24, 21, 57, 69, 51, 50, 51, 21, 19, 63, 91, 11, 6, 31, 63, 36, 39, 57, 47, 56, 65, 59, 4, 10, 12, 62, 43, 49, 54, 87, 29, 2, 18, 75, 39, 77, 69, 15, 78, 68, 37, 22, 41, 92, 67, 24, 87, 91, 31],
    [37, 16, 42, 47, 94, 14, 94, 34, 72, 36, 88, 51, 41, 71, 94, 99, 11, 97, 44, 77, 69, 91, 38, 25, 87, 7, 66, 54, 86, 49, 3, 48, 44, 93, 37, 82, 31, 59, 78, 33, 36, 3, 58, 10, 98, 6, 44, 62, 24, 94],
    [79, 93, 68, 75, 37, 44, 34, 39, 76, 62, 74, 28, 78, 43, 98, 83, 91, 27, 6, 82, 60, 44, 43, 76, 99, 66, 11, 35, 52, 8, 40, 62, 25, 24, 30, 1, 73, 27, 16, 91, 33, 11, 99, 2, 60, 90, 36, 62, 15, 3],
    [83, 87, 38, 38, 86, 67, 23, 19, 97, 78, 66, 67, 7, 23, 67, 8, 77, 71, 85, 29, 49, 3, 94, 76, 95, 48, 4, 37, 82, 57, 61, 6, 97, 5, 27, 95, 46, 92, 46, 52, 8, 11, 7, 54, 72, 57, 85, 22, 87, 65],
    [22, 29, 99, 25, 98, 55, 80, 82, 33, 68, 47, 74, 26, 61, 95, 55, 11, 42, 72, 14, 8, 98, 90, 36, 75, 69, 26, 24, 55, 98, 86, 30, 92, 94, 66, 47, 3, 41, 41, 47, 89, 28, 39, 80, 47, 57, 74, 38, 59, 5],
    [27, 92, 75, 94, 18, 41, 37, 58, 56, 20, 2, 39, 91, 81, 33, 14, 88, 22, 36, 65, 79, 23, 66, 5, 15, 51, 2, 81, 12, 40, 59, 32, 16, 87, 78, 41, 43, 94, 1, 93, 22, 93, 62, 53, 30, 34, 27, 30, 54, 77],
    [24, 47, 39, 66, 41, 46, 24, 23, 68, 50, 93, 22, 64, 81, 94, 97, 54, 82, 11, 91, 23, 32, 26, 22, 12, 23, 34, 87, 59, 2, 38, 84, 62, 10, 11, 93, 57, 81, 10, 40, 62, 49, 90, 34, 11, 81, 51, 21, 39, 27]
]).T

env = DummyVecEnv([lambda: GAEnv(processing_times)])
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=500000)

Npop = 50
generations = 2
num_elites = 50
start_time = time.time()
best_solution, best_obj, avg_obj,sol,c_max = HHGA_with_RL(Npop, generations, num_elites, processing_times, model, env)
elapsed_time = time.time() - start_time

print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("Elapsed Time:", elapsed_time)
print("sol:", sol)
print("cmax:", c_max)



Using cpu device
-----------------------------
| time/              |      |
|    fps             | 1319 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 953        |
|    iterations           | 2          |
|    time_elapsed         | 4          |
|    total_timesteps      | 4096       |
| train/                  |            |
|    approx_kl            | 0.00875466 |
|    clip_fraction        | 0.0394     |
|    clip_range           | 0.2        |
|    entropy_loss         | -1.61      |
|    explained_variance   | -0.019     |
|    learning_rate        | 0.0003     |
|    loss                 | 0.44       |
|    n_updates            | 10         |
|    policy_gradient_loss | -0.0052    |
|    value_loss           | 18         |
----------------------------------------
-----------------------------------

In [ ]:
numbers_str=""" 52 95 42 75 44 57 89 53 84 62 91 14 95 89  4 95  2 97 68 20 33 51 98  8 85 86 73  4 40 98 12 59 44 46  2 41 28 83 28 21 80 71  4 60 34 55 53 96 37 37 63 99 69 70 53 21 10 31 80 18  5 18 17 71 90 93 14 49 52  7 78 57 41 75 98 93 33 75 68 33 60 82 24 99  4 97 24 50 55 91 46 58 17 47 82  6 15 91 74 42 82 21 79 95 46 23 40 95 87 37 24 24 65 62 19 67 66  6 65 59  2 67 82 90 30 63  5 93 53 85 81 73 34 74 13 78 35 20 16 48 12 11 80  9 24 76 32 35 66 48 16 26 46 66 76 31 36  8 37 21  3 76 67  5 47 72 66 56 95 49 47 26 81 56 76 66 36 53 26 52 29 36 68 21 71 61 71 69 28 86 27 41 86 55 17 62 96 59 53 93 63 55 59 35 21 59 78 25 30 38 78 79 58 44 38 76 70 72 85  8 10 84 42 67 20 24 75 23 33 60 20 75 83 26 92 29 39 14 74 66 86 10 27  8  7 97 84 56 61  9 94 34 89 62 47 66 76 15 18 54 24 55 96 10 12 96 53 92 77  6 91 14 41 30 85 17 23 60 76 39 85 10 65 15 55 41 28 93 88 27 77 81 19 76 55 67 65  8 18 56 79 21 93 32  8 45 37 78 26 98 17 25 21 28 68 24 62 89 60 64 38 90 87  1 99 34  9 22 74 14 14 84 75 37 32 29 32 89 12 47 19 97  7 12 43 89 14 33 56 57 22  6 24 55 48 57 78  5 50 83 70 21 71 58 36 50 31 86 29 30 93 49 83 89 44 38 62 45 22 85 39 98 56 68 84 77 67 53 46 24 52 96  2 88 33 27 49 78 82 65 80 13 64 77 17 78 82  4 72 93 68 25 67 80 43 93 21 33 14 30 59 83 85 85 70 35  2 76 46 72 69 46  3 57 71 77 33 49 59 82 59 70 76 10 65 19 77 86 21 75 96  3 50 57 66 84 98 55 70 32 31 64 11  9 32 58 98 95 25  4 45 60 87 31  1 96 22 95 73 77 30 88 14 22 93 48 10  7 14 91  5 43 30 79 39 34 77 81 11 10 53 19 99 62 88 93 34 72 42 65 39 79  9 26 72 29 36 48 57 95 93 79 88 77 94 39 74 46 17 30 62 77 43 98 48 14 45 25 98 30 90 92 35 13 75 55 80 67  3 93 54 67 25 77 38 98 96 20 15 36 65 97 27 25 61 24 97 61 75 92 73 21 29  3 96 51 26 44 56 31 64 38 44 46 66 31 48 27 82 51 90 63 85 36 69 67 81 18 81 72 71 90 59 82 22 88 35 49 78 69 76  2 14  3 22 26 44  1  4 16 55 43 87 35 76 98 78 81 48 25 81 27 84 59 98 14 32 95 30 13 68 19 57 65 13 63 26 96 53 94 27 93 49 63 65 34 10 56 51 97 52 46 16 50 96 85 61 76 30 90 42 88 37 43 88 91 14 63 65 74 71  8 39 95 82 17 38 69 17 24 66 75 52 59  4 73 56 19 39 51 95 53 54 22 84 54  2 80 84 66 25 16 79 90 51 29 29 90 83 83 19 95 87 12 34 23 44 30 82 83 42 56 89 38 96 10  3 53 97 11 65 47 76 22 17 14 11 69 91 53  3 80 78 32 53 43 85 19 48 49 66 22 37 51 82 59 88 77 19 32 52  9 96 23 64 22 37  3 52 44 11 21 85  6 40 68 30 35 58 31 11 11  6 59 64 65 23 80 75 63 92 62 11 83 87 66 98 42 23 45 52  6  3 64 55 97 83 42 81 92 68 46 56 88 50 13 23 13 49 18 50 94 71 64 31 21  2 63 58 36 64 52  8 94 51 36 82 30 17 21 80 38 55 34 85 44 47 66 19 66 61 60 98 82 79 71 28 74 27 33 13  9 12 51 16 49 83 48 13 78 96 77 68 88 77 76 73 92 72 87 66 98 40 31 75 45 98 90  4 23 61 86 16 42 14 92 67 77 46 41 78  3 72 95 53 59 34 66 42 63 27 92  8 65 34  6 42 39  2  7 85 32 14 74 59 95 48 37 59  4 42 93 32 30 16 95 58 12 95 21 74 38  4 31 62 39 97 57  9 54 13 47  6 70 19 97 41  1 57 60 62 14 90 76 12 89 37 35 91 69 55 48 56 84 22 51 43 50 62 61 10 87 99 40 91 64 62 53 33 16 """

In [ ]:
# Split the string into individual numbers
numbers_list = numbers_str.split()
print(numbers_list)
# Convert the numbers to integers
numbers = [int(num) for num in numbers_list]

# Reshape the list of numbers into a 100x10 NumPy array
np_table = np.array(numbers).reshape(100, 10)

print(np_table)


['52', '95', '42', '75', '44', '57', '89', '53', '84', '62', '91', '14', '95', '89', '4', '95', '2', '97', '68', '20', '33', '51', '98', '8', '85', '86', '73', '4', '40', '98', '12', '59', '44', '46', '2', '41', '28', '83', '28', '21', '80', '71', '4', '60', '34', '55', '53', '96', '37', '37', '63', '99', '69', '70', '53', '21', '10', '31', '80', '18', '5', '18', '17', '71', '90', '93', '14', '49', '52', '7', '78', '57', '41', '75', '98', '93', '33', '75', '68', '33', '60', '82', '24', '99', '4', '97', '24', '50', '55', '91', '46', '58', '17', '47', '82', '6', '15', '91', '74', '42', '82', '21', '79', '95', '46', '23', '40', '95', '87', '37', '24', '24', '65', '62', '19', '67', '66', '6', '65', '59', '2', '67', '82', '90', '30', '63', '5', '93', '53', '85', '81', '73', '34', '74', '13', '78', '35', '20', '16', '48', '12', '11', '80', '9', '24', '76', '32', '35', '66', '48', '16', '26', '46', '66', '76', '31', '36', '8', '37', '21', '3', '76', '67', '5', '47', '72', '66', '56', '95', '4

In [ ]:
# Define the processing times matrix
processing_times = np_table

env = DummyVecEnv([lambda: GAEnv(processing_times)])
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=500000)

Npop = 50
generations = 2
num_elites = 50
start_time = time.time()
best_solution, best_obj, avg_obj,sol,c_max = HHGA_with_RL(Npop, generations, num_elites, processing_times, model, env)
elapsed_time = time.time() - start_time

print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("Elapsed Time:", elapsed_time)
print("sol:", sol)
print("cmax:", c_max)



/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Using cpu device
-----------------------------
| time/              |      |
|    fps             | 1276 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
------------------------------------------
| time/                   |              |
|    fps                  | 788          |
|    iterations           | 2            |
|    time_elapsed         | 5            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0073854146 |
|    clip_fraction        | 0.0208       |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.61        |
|    explained_variance   | -0.03        |
|    learning_rate        | 0.0003       |
|    loss                 | 0.374        |
|    n_updates            | 10           |
|    policy_gradient_loss | -0.00278     |
|    value_loss           | 17.9         |
------------------------------------------

In [ ]:
# Define the processing times matrix
processing_times = np.array([
    [46, 52, 79, 45, 97, 10, 44, 24, 85, 75, 66, 49, 95, 61, 19, 47, 84, 13, 11, 19, 98, 2, 85, 44, 7, 73, 19, 69, 12, 73, 85, 23, 53, 16, 88, 8, 26, 42, 58, 63, 7, 2, 44, 38, 24, 76, 85, 61, 32, 90],
    [61, 87, 51, 25, 73, 93, 28, 90, 94, 59, 64, 2, 16, 35, 53, 40, 81, 26, 85, 4, 4, 10, 63, 96, 55, 71, 66, 94, 7, 15, 11, 99, 37, 50, 56, 69, 22, 56, 67, 63, 96, 74, 4, 42, 40, 30, 93, 36, 25, 87],
    [3, 1, 58, 85, 33, 71, 58, 56, 64, 43, 48, 69, 96, 35, 82, 53, 64, 11, 61, 36, 53, 87, 88, 10, 32, 38, 25, 24, 90, 7, 11, 49, 2, 76, 17, 32, 39, 9, 83, 69, 67, 28, 88, 23, 91, 71, 3, 26, 41, 96],
    [51, 24, 21, 57, 69, 51, 50, 51, 21, 19, 63, 91, 11, 6, 31, 63, 36, 39, 57, 47, 56, 65, 59, 4, 10, 12, 62, 43, 49, 54, 87, 29, 2, 18, 75, 39, 77, 69, 15, 78, 68, 37, 22, 41, 92, 67, 24, 87, 91, 31],
    [37, 16, 42, 47, 94, 14, 94, 34, 72, 36, 88, 51, 41, 71, 94, 99, 11, 97, 44, 77, 69, 91, 38, 25, 87, 7, 66, 54, 86, 49, 3, 48, 44, 93, 37, 82, 31, 59, 78, 33, 36, 3, 58, 10, 98, 6, 44, 62, 24, 94],
    [79, 93, 68, 75, 37, 44, 34, 39, 76, 62, 74, 28, 78, 43, 98, 83, 91, 27, 6, 82, 60, 44, 43, 76, 99, 66, 11, 35, 52, 8, 40, 62, 25, 24, 30, 1, 73, 27, 16, 91, 33, 11, 99, 2, 60, 90, 36, 62, 15, 3],
    [83, 87, 38, 38, 86, 67, 23, 19, 97, 78, 66, 67, 7, 23, 67, 8, 77, 71, 85, 29, 49, 3, 94, 76, 95, 48, 4, 37, 82, 57, 61, 6, 97, 5, 27, 95, 46, 92, 46, 52, 8, 11, 7, 54, 72, 57, 85, 22, 87, 65],
    [22, 29, 99, 25, 98, 55, 80, 82, 33, 68, 47, 74, 26, 61, 95, 55, 11, 42, 72, 14, 8, 98, 90, 36, 75, 69, 26, 24, 55, 98, 86, 30, 92, 94, 66, 47, 3, 41, 41, 47, 89, 28, 39, 80, 47, 57, 74, 38, 59, 5],
    [27, 92, 75, 94, 18, 41, 37, 58, 56, 20, 2, 39, 91, 81, 33, 14, 88, 22, 36, 65, 79, 23, 66, 5, 15, 51, 2, 81, 12, 40, 59, 32, 16, 87, 78, 41, 43, 94, 1, 93, 22, 93, 62, 53, 30, 34, 27, 30, 54, 77],
    [24, 47, 39, 66, 41, 46, 24, 23, 68, 50, 93, 22, 64, 81, 94, 97, 54, 82, 11, 91, 23, 32, 26, 22, 12, 23, 34, 87, 59, 2, 38, 84, 62, 10, 11, 93, 57, 81, 10, 40, 62, 49, 90, 34, 11, 81, 51, 21, 39, 27]
]).T

env = DummyVecEnv([lambda: GAEnv(processing_times)])
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=500000)

Npop = 50
generations = 2
num_elites = 50
start_time = time.time()
best_solution, best_obj, avg_obj,sol,c_max = HHGA_with_RL(Npop, generations, num_elites, processing_times, model, env)
elapsed_time = time.time() - start_time

print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("Elapsed Time:", elapsed_time)
print("sol:", sol)
print("cmax:", c_max)



In [ ]:
# Assuming you have already defined the processing_times matrix and the function recherche_tabou

# Define the parameters
tabu_size = 14
max_it = 169
max_it_stagn = 1719
init_funct = [4]  # Make sure it's an array
neighbor_generation = 1

# Initialize the sequence (you can use any initial sequence)
num_jobs = processing_times.shape[0]
seq = list(range(num_jobs))

# Call the recherche_tabou function
best_seq, best_cmax = recherche_tabou(processing_times, seq, tabu_size, max_it, max_it_stagn, neighbor_generation)

# best_seq will contain the optimized sequence of jobs
print("Optimized Sequence:", best_seq)
print("Objective Function Value:", best_cmax)


Optimized Sequence: [16, 1, 15, 8, 11, 5, 3, 2, 0, 6, 4, 18, 13, 14, 9, 17, 7, 19, 12, 10]
Objective Function Value: 1231.0


In [ ]:
# Define the processing times matrix
processing_times = np.array([
    [34, 20, 57, 47, 62, 40, 74, 94,  9, 62, 86, 13, 78, 46, 83, 52, 13, 70, 40, 60],
    [ 5, 48, 80, 43, 34,  2, 87, 68, 28, 84, 30, 35, 42, 39, 85, 34, 36,  9, 96, 84],
    [86, 35,  5, 93, 74, 12, 40, 95, 80,  6, 92, 14, 83, 49, 36, 38, 43, 89, 94, 33],
    [28, 39, 55, 21, 25, 88, 59, 40, 90, 18, 33, 10, 59, 92, 15, 77, 31, 85, 85, 99],
    [ 8, 91, 45, 55, 75, 18, 59, 86, 45, 89, 11, 54, 38, 41, 64, 98, 83, 36, 61, 19]
]).T

env = DummyVecEnv([lambda: GAEnv(processing_times)])
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=100000)

Npop = 100
generations = 2
num_elites = 50
start_time = time.time()
best_solution, best_obj, avg_obj = HHGA_with_RL(Npop, generations, num_elites, processing_times, model, env)
elapsed_time = time.time() - start_time

print("Best Solution:", best_solution)
print("Best Objective Function Value:", best_obj)
print("Average Objective Function Value:", avg_obj)
print("Elapsed Time:", elapsed_time)

Using cpu device
-----------------------------
| time/              |      |
|    fps             | 760  |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 681         |
|    iterations           | 2           |
|    time_elapsed         | 6           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008218061 |
|    clip_fraction        | 0.0382      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.6        |
|    explained_variance   | -0.0294     |
|    learning_rate        | 0.0003      |
|    loss                 | 0.425       |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00504    |
|    value_loss           | 17.8        |
-----------------------------------------
-----------------

In [ ]:
# Assuming you have already defined the processing_times matrix and the function recherche_tabou

# Define the parameters
tabu_size = 18
max_it = 914
max_it_stagn = 1637
init_funct = [4]  # Make sure it's an array
neighbor_generation = 2

# Initialize the sequence (you can use any initial sequence)
num_jobs = processing_times.shape[0]
seq = list(range(num_jobs))

# Call the recherche_tabou function
best_seq, best_cmax = recherche_tabou(processing_times, seq, tabu_size, max_it, max_it_stagn, neighbor_generation)

# best_seq will contain the optimized sequence of jobs
print("Optimized Sequence:", best_seq)
print("Objective Function Value:", best_cmax)


Optimized Sequence: [16, 1, 15, 8, 11, 3, 5, 18, 7, 9, 19, 17, 6, 4, 12, 13, 14, 0, 2, 10]
Objective Function Value: 1211.0


In [ ]:
# Assuming you have already defined the processing_times matrix and the function recherche_tabou

# Define the parameters
tabu_size = 18
max_it = 914
max_it_stagn = 1637
init_funct = [4]  # Make sure it's an array
neighbor_generation = 2

# Initialize the sequence (you can use any initial sequence)
num_jobs = processing_times.shape[0]
seq = list(range(num_jobs))
bb=100000
tt=[]
# Call the recherche_tabou function
for i in range(20)   :
    best_seq, best_cmax = recherche_tabou(processing_times, seq, tabu_size, max_it, max_it_stagn, neighbor_generation)
    if bb>best_cmax :
       bb=best_cmax
       tt=best_seq

# best_seq will contain the optimized sequence of jobs
print("Optimized Sequence:", tt)
print("Objective Function Value:", bb)


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Optimized Sequence: [16, 1, 15, 8, 3, 2, 13, 7, 18, 6, 0, 9, 5, 11, 17, 14, 12, 4, 19, 10]
Objective Function Value: 1214.0
